# 49. 실제 Segmentation 데이터셋 준비

7장은 빠르게 학습하고, 결과를 확실하게 보고, 여러 개선 실험을 비교하는 데 집중합니다.

이 노트북에서는 외부 다운로드 없이 로컬에 작은 image-mask 파일 데이터셋을 만듭니다. 데이터는 단순하지만 실제 프로젝트처럼 `images/train`, `masks/train`, `images/val`, `masks/val` 구조를 갖기 때문에 이후 노트북에서 동일한 파이프라인을 사용할 수 있습니다.

이번 노트북의 목표는 다음과 같습니다.

- 학습이 빠른 segmentation 데이터셋을 로컬 파일로 생성합니다.
- class id, pixel 비율, train/validation 개수를 점검합니다.
- 56장에서 비교할 수 있도록 데이터셋 점검 산출물을 저장합니다.

In [ ]:
from pathlib import Path
import sys

NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / "seg7_utils.py").exists():
    NOTEBOOK_DIR = Path("Vision 기초/7장")

sys.path.append(str(NOTEBOOK_DIR))
DATA_ROOT = NOTEBOOK_DIR / "data" / "mini_shapes_seg"
RUNS_ROOT = NOTEBOOK_DIR / "runs"

from seg7_utils import *
set_korean_font()
set_seed(7)

## 49-1. MiniShapesSeg 데이터셋 생성

In [ ]:
dataset_root = create_mini_shapes_dataset(
    DATA_ROOT,
    n_train=96,
    n_val=24,
    image_size=64,
    seed=7,
)

print(dataset_root)
read_json(dataset_root / "meta.json")

## 49-2. 데이터셋 구조와 class 비율 확인

In [ ]:
summary = inspect_dataset(DATA_ROOT)
summary

## 49-3. 샘플 image-mask 확인과 산출물 저장

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np

run_dir = RUNS_ROOT / "49_dataset_check"
(run_dir / "pred_samples").mkdir(parents=True, exist_ok=True)

pairs = list_pairs(DATA_ROOT, "train")[:4]
fig, axes = plt.subplots(len(pairs), 3, figsize=(8, 8))
for row, (image_path, mask_path) in enumerate(pairs):
    image = np.asarray(Image.open(image_path).convert("RGB"))
    mask = np.asarray(Image.open(mask_path))
    axes[row, 0].imshow(image)
    axes[row, 0].set_title("image")
    axes[row, 1].imshow(colorize_mask(mask))
    axes[row, 1].set_title("mask")
    axes[row, 2].imshow(make_overlay(image, mask))
    axes[row, 2].set_title("overlay")
    for ax in axes[row]:
        ax.axis("off")
fig.tight_layout()
fig.savefig(run_dir / "pred_samples" / "dataset_samples.png", dpi=140)
plt.show()

save_json(run_dir / "config.json", {
    "experiment_name": "49_dataset_check",
    "dataset_root": str(DATA_ROOT),
    "note": "dataset generation and sanity check",
})
save_json(run_dir / "metrics.json", {
    "experiment_name": "49_dataset_check",
    "dataset_summary": summary,
    "pixel_accuracy": None,
    "mean_iou": None,
})